# 01. 데이터 준비 + 임베딩 (한 번만, 드라이브 소유자 계정으로)

공유 폴더 소유자(결제한 계정)가 한 번만 실행한다. 원본은 Colab 로컬 디스크에만 받고, 드라이브에는 CSV 2개와 축소 이미지 zip, 가공 파일만 올린다.

준비: Colab 왼쪽 🔑(보안 비밀)에 `kaggle_username`, `kaggle_key` 등록 (새 방식 KGAT_ 토큰도 가능), H&M 대회 규칙 동의. 런타임은 GPU.

In [ ]:
# @title 0. 공통 준비 (모든 노트북 동일)
from google.colab import drive
drive.mount("/content/drive")

REPO_URL = "https://github.com/yuhwani/multimodal-recsys.git"  # upstream에 머지되면 3sunghynix 주소로 변경
BRANCH = "main"
!rm -rf /content/repo && git clone -q -b {BRANCH} {REPO_URL} /content/repo
!pip -q install -r /content/repo/paper/requirements.txt
!cd /content/repo && git log -1 --format="코드 커밋: %h %s (%cd)"

import sys
sys.path.insert(0, "/content/repo/paper")
from msrs_paper.config import load_all, Paths
cfg, exps, plan = load_all()
DRIVE_ROOT = cfg["drive_root"]  # 공유 폴더 바로가기 경로가 다르면 이 줄만 바꾼다
paths = Paths(DRIVE_ROOT).make_dirs()
print("드라이브 폴더:", paths.root)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# @title 1. Kaggle에서 원본 zip 받기 (Colab 로컬 디스크, 드라이브 아님)
import os, glob
from google.colab import userdata

def secret(*names):
    for n in names:
        try:
            v = userdata.get(n)
        except Exception:
            continue
        if v:
            return v.strip().strip('"').strip("'")
    raise KeyError(f"보안 비밀 {names} 없음")

user = secret("kaggle_username", "KAGGLE_USERNAME")
key = secret("kaggle_key", "KAGGLE_KEY")
for k in ("KAGGLE_USERNAME", "KAGGLE_KEY", "KAGGLE_API_TOKEN"):
    os.environ.pop(k, None)
if key.startswith("KGAT_"):                 # 새 방식 API 토큰
    os.environ["KAGGLE_API_TOKEN"] = key
else:                                        # 기존 kaggle.json 방식
    os.environ["KAGGLE_USERNAME"] = user
    os.environ["KAGGLE_KEY"] = key
print("사용자:", user, "| 키 형식:", "새 API 토큰" if key.startswith("KGAT_") else "기존 키", "| 길이:", len(key))

!pip -q install -U kaggle
!kaggle competitions files -c h-and-m-personalized-fashion-recommendations | head -5   # 로그인 확인
!df -h /content | tail -1
!mkdir -p /content/hm && kaggle competitions download -c h-and-m-personalized-fashion-recommendations -p /content/hm
ZIP = glob.glob("/content/hm/*.zip")[0]
print(ZIP, round(os.path.getsize(ZIP) / 1e9, 1), "GB")

In [ ]:
# @title 2. CSV 2개만 드라이브 raw/ 로 풀기
!unzip -o -j "{ZIP}" articles.csv transactions_train.csv -d "{paths.raw}"
!ls -lh "{paths.raw}"

In [ ]:
# @title 3. 컬럼 값 확인 → analysis_plan.yaml 확정용
import pandas as pd
from msrs_paper import data_prep as dp
articles = pd.read_csv(paths.raw / "articles.csv", dtype={"article_id": "int64"})
for col in ["graphical_appearance_name", "product_group_name"]:
    print(articles[col].value_counts().to_string(), "\n")
problems = dp.check_plan(articles, plan)
print("매핑에 없는 값:", problems or "없음")
print("analysis_plan status:", plan["status"])

**여기서 멈추고** `paper/configs/analysis_plan.yaml`을 위 값으로 확정 → `status: final` → 커밋·푸시한 뒤, 0번 셀부터 다시 실행한다 (1·2번은 파일이 있으면 건너뛰어도 됨). 결과를 보기 전에 확정해야 사전 등록이 된다.

In [ ]:
# @title 4. 상품 표 + 분할 (main, alt)
import numpy as np
from msrs_paper import embeddings as em
from msrs_paper.utils import write_json
assert plan["status"] == "final", "analysis_plan.yaml을 확정(final)·커밋하고 0번 셀부터 다시 실행하세요"
assert not dp.check_plan(articles, plan)
image_ids = em.image_ids_in_zip(ZIP)
tx = dp.load_transactions(paths.raw / "transactions_train.csv")
print("구매 기록", f"{len(tx):,}", tx["t_dat"].min().date(), "~", tx["t_dat"].max().date())
items = dp.select_items(tx, dp.build_item_table(articles, plan), image_ids,
                        cfg["data"]["window_weeks"], cfg["data"]["alt_shift_weeks"])
items.to_parquet(paths.processed / "items.parquet", index=False)
items[["idx", "article_id"]].to_csv(paths.embeddings / "item_index.csv", index=False)
print("상품", len(items))
for split, shift in (("main", 0), ("alt", cfg["data"]["alt_shift_weeks"])):
    history, vt, tt, stats = dp.build_split(tx, items, cfg["data"]["window_weeks"], cfg["data"]["min_user_purchases"], shift)
    dp.save_split(paths.split_dir(split), history, vt, tt, stats)
    print(split, {k: v for k, v in stats.items() if k != "test_cells"})

In [ ]:
# @title 5. 필요한 상품 이미지만 축소해서 zip으로 (로컬 → 드라이브 복사)
IMG_ZIP = "/content/images_subset.zip"
em.write_image_subset(ZIP, items["article_id"].values, IMG_ZIP, cfg["data"]["image_max_side"])
!cp "{IMG_ZIP}" "{paths.raw}/images_subset.zip" && ls -lh "{paths.raw}"

In [ ]:
# @title 6. 임베딩 추출 → PCA → L2 정규화 (이미 있는 파일은 건너뜀, 런타임이 끊긴 뒤에도 이 셀부터 가능)
import os, shutil, torch
import numpy as np, pandas as pd
from msrs_paper import embeddings as em
IMG_ZIP = "/content/images_subset.zip"
if not os.path.exists(IMG_ZIP):
    shutil.copy(paths.raw / "images_subset.zip", IMG_ZIP)
items = pd.read_parquet(paths.processed / "items.parquet").sort_values("idx").reset_index(drop=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
E = cfg["embedding"]
if not (paths.embeddings / "text_minilm.npy").exists():
    raw = em.extract_text(items["text"].values, E["text_model"], 256, device)
    emb, info = em.reduce(raw, E["pca_dim"])
    em.save_embedding(paths, "text_minilm", emb, {**info, "model": E["text_model"]})
for name, spec in E["image_models"].items():
    if (paths.embeddings / f"img_{name}.npy").exists():
        continue
    raw = em.extract_image(IMG_ZIP, items["article_id"].values, spec, E["batch_size"], device)
    emb, info = em.reduce(raw, E["pca_dim"])
    em.save_embedding(paths, f"img_{name}", emb, {**info, **spec})
    print(name, emb.shape, info)

In [ ]:
# @title 7. 섞기 대응표 (전체 / 같은 품목 안) × seed
from msrs_paper.utils import write_json
for s in E["shuffle_seeds"]:
    g, t, info = em.make_shuffles(items["product_type_name"].values, s)
    np.save(paths.embeddings / f"shuffle_global_seed{s}.npy", g)
    np.save(paths.embeddings / f"shuffle_type_seed{s}.npy", t)
    write_json(paths.embeddings / f"shuffle_seed{s}.json", info)
    print(info)

In [ ]:
# @title 8. 임베딩 품질 눈으로 확인 (첫 열이 기준 상품, 나머지가 가까운 상품)
from IPython.display import Image, display
from msrs_paper.inspect_images import neighbor_grid
checks = paths.results / "checks"; checks.mkdir(parents=True, exist_ok=True)
for name in ["text_minilm"] + [f"img_{n}" for n in E["image_models"]]:
    out = neighbor_grid(items, np.load(paths.embeddings / f"{name}.npy"), IMG_ZIP, checks / f"neighbors_{name}.png")
    print(name); display(Image(str(out)))

In [ ]:
# @title 9. 체크섬 목록 작성 (이후 모든 노트북이 이걸로 같은 준비물인지 확인)
from msrs_paper.utils import write_manifest
m = write_manifest(paths)
print(len(m["files"]), "개 파일 등록 완료")